In [57]:
import os
import ffmpeg
import openai
import gradio as gr

openai.api_key = os.getenv("OPENAI_API_KEY")
# Cache simple de transcripciones: {nombre_archivo: texto_transcrito}
transcripciones_cache = {}


In [ ]:
def extraer_audio_con_ffmpeg(video_path, output_audio_path="audio_extraido.wav"):
    try:
        (
            ffmpeg
            .input(video_path)
            .output(output_audio_path, format='wav', acodec='pcm_s16le', ac=1, ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )
        return output_audio_path
    except ffmpeg.Error as e:
        return None

def transcribir_audio(file_path):
    with open(file_path, "rb") as f:
        transcript = openai.audio.transcriptions.create(
            model="whisper-1",
            file=f
        )
    return transcript.text

def responder_con_gpt4(pregunta, contexto):
    prompt = f"""
Eres un asistente educativo que responde en formato Markdown. Usa listas, bloques de código y **fórmulas matemáticas con doble signo de dólar `$$`** para que se rendericen correctamente.

--- CONTEXTO TRANSCRITO ---
{contexto}

--- PREGUNTA DEL USUARIO ---
{pregunta}

Responde usando **Markdown** con formato matemático en bloque usando `$$`.
"""
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800
    )
    return response.choices[0].message.content


In [59]:
def interfaz_gradio(archivo, pregunta_usuario):
    if archivo is None:
        return "⚠️ No se proporcionó ningún archivo.", ""
    
    ext = os.path.splitext(archivo)[1].lower()

    if ext in [".mp3", ".wav", ".m4a"]:
        archivo_audio = archivo
    elif ext in [".mp4", ".mov", ".avi", ".mkv"]:
        archivo_audio = extraer_audio_con_ffmpeg(archivo)
        if archivo_audio is None:
            return "⚠️ No se pudo extraer el audio del video.", ""
    else:
        return f"⚠️ Tipo de archivo no soportado: {ext}", ""

    transcripcion = transcribir_audio(archivo_audio)

    if not pregunta_usuario.strip():
        return transcripcion, "💡 Escribe una pregunta para recibir una respuesta."

    respuesta = responder_con_gpt4(pregunta_usuario, transcripcion)
    
    return transcripcion, respuesta


In [62]:
with gr.Blocks(title="Asistente Multimodal con GPT-4") as demo:
    gr.Markdown("## 🎓 Asistente Multimodal con GPT-4")
    gr.Markdown("Sube un archivo de audio o video, haz una pregunta, y GPT-4 responderá con soporte para código, listas y fórmulas matemáticas en Markdown.")
    
    archivo = gr.File(label="📁 Archivo de entrada (audio o video)", file_types=[".mp3", ".wav", ".m4a", ".mp4", ".mov", ".avi", ".mkv"])
    pregunta = gr.Textbox(label="❓ Pregunta", placeholder="¿Qué explica el profesor?", lines=1)
    
    estado = gr.Markdown("🟡 Esperando entrada del usuario...")
    salida = gr.Markdown(label="🤖 Respuesta del asistente")

    def pipeline(archivo, pregunta):
        if archivo is None:
            yield "⚠️ Por favor sube un archivo.", ""
            return
        
        nombre_archivo = os.path.basename(archivo)
        ext = os.path.splitext(nombre_archivo)[1].lower()

        yield "🔄 Procesando archivo...", ""

        if ext in [".mp3", ".wav", ".m4a"]:
            archivo_audio = archivo
        elif ext in [".mp4", ".mov", ".avi", ".mkv"]:
            archivo_audio = extraer_audio_con_ffmpeg(archivo)
            if archivo_audio is None:
                yield "❌ No se pudo extraer el audio del video.", ""
                return
        else:
            yield f"❌ Tipo de archivo no soportado: {ext}", ""
            return

        # ✅ Usar transcripción cacheada si ya existe
        if nombre_archivo in transcripciones_cache:
            transcripcion = transcripciones_cache[nombre_archivo]
            yield "✅ Transcripción recuperada de memoria.", ""
        else:
            yield "🔊 Transcribiendo el contenido con Whisper...", ""
            transcripcion = transcribir_audio(archivo_audio)
            transcripciones_cache[nombre_archivo] = transcripcion  # Guardar en cache

        yield "🤖 Consultando a GPT-4...", ""
        respuesta = responder_con_gpt4(pregunta, transcripcion)

        yield "✅ Listo. Aquí tienes la respuesta:", respuesta


    submit_btn = gr.Button("Enviar")
    submit_btn.click(pipeline, inputs=[archivo, pregunta], outputs=[estado, salida])

demo.launch()


* Running on local URL:  http://127.0.0.1:7869

To create a public link, set `share=True` in `launch()`.
